In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("NA")
    else:
        df[col] = df[col].fillna(0.0)

In [4]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [ ]:
Question 1: ROC AUC feature importance

In [5]:
from sklearn.metrics import roc_auc_score

numerical = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

y_train = df_train.converted

for col in numerical:
    auc = roc_auc_score(y_train, df_train[col])

    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])

    print(col, round(auc, 3))

lead_score 0.788
number_of_courses_viewed 0.723
interaction_count 0.765
annual_income 0.608


In [ ]:
Question 2: Train logistic regression

In [6]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

y_train = df_train.converted
y_val = df_val.converted

train_dict = df_train.drop("converted", axis=1).to_dict(
    orient="records"
)
val_dict = df_val.drop("converted", axis=1).to_dict(
    orient="records"
)

dv = DictVectorizer()

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [ ]:
Train the Model

In [7]:
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000
)

model.fit(X_train, y_train)


LogisticRegression(max_iter=1000, solver='liblinear')

In [ ]:
Calculate AUC on the validation data:

In [8]:
y_pred = model.predict_proba(X_val)[:, 1]

roc_auc_score(y_val, y_pred)

np.float64(0.7318222288173319)

In [ ]:
Question 3: Precision and recall
Try thresholds from 0 to 1, increasing by 0.01.

In [9]:
from sklearn.metrics import precision_score, recall_score

for t in np.arange(0, 1.01, 0.01):
    y_threshold = (y_pred >= t).astype(int)

    precision = precision_score(
        y_val, y_threshold, zero_division=0
    )
    recall = recall_score(
        y_val, y_threshold, zero_division=0
    )

    print(round(t, 2), precision, recall)

0.0 0.586 1.0
0.01 0.586 1.0
0.02 0.586 1.0
0.03 0.586 1.0
0.04 0.586 1.0
0.05 0.586 1.0
0.06 0.586 1.0
0.07 0.586 1.0
0.08 0.586 1.0
0.09 0.586 1.0
0.1 0.586 1.0
0.11 0.586 1.0
0.12 0.586 1.0
0.13 0.586 1.0
0.14 0.586 1.0
0.15 0.586 1.0
0.16 0.586 1.0
0.17 0.586 1.0
0.18 0.586 1.0
0.19 0.586 1.0
0.2 0.586 1.0
0.21 0.586 1.0
0.22 0.586 1.0
0.23 0.586 1.0
0.24 0.586 1.0
0.25 0.586 1.0
0.26 0.5865865865865866 1.0
0.27 0.5877632898696088 1.0
0.28 0.591321897073663 1.0
0.29 0.591321897073663 1.0
0.3 0.5925176946410515 1.0
0.31 0.5963302752293578 0.9982935153583617
0.32 0.5961145194274029 0.9948805460750854
0.33 0.5973360655737705 0.9948805460750854
0.34 0.6016511867905057 0.9948805460750854
0.35 0.6028955532574974 0.9948805460750854
0.36 0.6047717842323651 0.9948805460750854
0.37 0.6069182389937107 0.9880546075085325
0.38 0.6073684210526316 0.984641638225256
0.39 0.6095238095238096 0.9829351535836177
0.4 0.6128342245989304 0.9778156996587031
0.41 0.6190476190476191 0.9761092150170648
0.42 

In [ ]:
Question 4: F1 score
F1 combines precision and recall into one score.

In [10]:
from sklearn.metrics import f1_score

for t in np.arange(0, 1.01, 0.01):
    y_threshold = (y_pred >= t).astype(int)

    f1 = f1_score(y_val, y_threshold)

    print(round(t, 2), f1)

0.0 0.7389659520807061
0.01 0.7389659520807061
0.02 0.7389659520807061
0.03 0.7389659520807061
0.04 0.7389659520807061
0.05 0.7389659520807061
0.06 0.7389659520807061
0.07 0.7389659520807061
0.08 0.7389659520807061
0.09 0.7389659520807061
0.1 0.7389659520807061
0.11 0.7389659520807061
0.12 0.7389659520807061
0.13 0.7389659520807061
0.14 0.7389659520807061
0.15 0.7389659520807061
0.16 0.7389659520807061
0.17 0.7389659520807061
0.18 0.7389659520807061
0.19 0.7389659520807061
0.2 0.7389659520807061
0.21 0.7389659520807061
0.22 0.7389659520807061
0.23 0.7389659520807061
0.24 0.7389659520807061
0.25 0.7389659520807061
0.26 0.7394321766561515
0.27 0.7403663929248263
0.28 0.7431832593532023
0.29 0.7431832593532023
0.3 0.7441269841269841
0.31 0.7466496490108487
0.32 0.7455242966751918
0.33 0.7464788732394366
0.34 0.74983922829582
0.35 0.7508048937540245
0.36 0.752258064516129
0.37 0.7519480519480519
0.38 0.7513020833333334
0.39 0.752449379490529
0.4 0.7534516765285996
0.41 0.7576158940397351
0

In [ ]:
Question 5: 5-fold cross-validation
Use five different splits to check how consistently the model performs.

In [11]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

kfold = KFold(
    n_splits=5, shuffle=True, random_state=1
)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train_fold = df_full_train.iloc[train_idx]
    df_val_fold = df_full_train.iloc[val_idx]

    y_train_fold = df_train_fold.converted
    y_val_fold = df_val_fold.converted

    dv = DictVectorizer()

    X_train_fold = dv.fit_transform(
        df_train_fold.drop("converted", axis=1).to_dict(
            orient="records"
        )
    )

    X_val_fold = dv.transform(
        df_val_fold.drop("converted", axis=1).to_dict(
            orient="records"
        )
    )

    model = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000
    )

    model.fit(X_train_fold, y_train_fold)

    y_pred_fold = model.predict_proba(X_val_fold)[:, 1]

    auc = roc_auc_score(y_val_fold, y_pred_fold)
    scores.append(auc)

print(scores)
print(np.std(scores))

[np.float64(0.7305601027122453), np.float64(0.7295726148196622), np.float64(0.7363428071259397), np.float64(0.7412454717957564), np.float64(0.7209138590326581)]
0.006853233151022138


In [ ]:
Question 6: Hyperparameter tuning

In [12]:
for C in [0.000001, 0.001, 1]:
    scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_fold = df_full_train.iloc[train_idx]
        df_val_fold = df_full_train.iloc[val_idx]

        y_train_fold = df_train_fold.converted
        y_val_fold = df_val_fold.converted

        dv = DictVectorizer()

        X_train_fold = dv.fit_transform(
            df_train_fold.drop("converted", axis=1).to_dict(
                orient="records"
            )
        )

        X_val_fold = dv.transform(
            df_val_fold.drop("converted", axis=1).to_dict(
                orient="records"
            )
        )

        model = LogisticRegression(
            solver="liblinear", C=C, max_iter=1000
        )

        model.fit(X_train_fold, y_train_fold)

        y_pred_fold = model.predict_proba(X_val_fold)[:, 1]

        scores.append(
            roc_auc_score(y_val_fold, y_pred_fold)
        )

    print(C, round(np.mean(scores), 3), round(np.std(scores), 3))

1e-06 0.617 0.019
0.001 0.749 0.01
1 0.732 0.007
